# Modelado Predictivo de la Infracción C02

Se desarrolla un modelo predictivo para la infracción C02 (estacionamiento en sitios prohibidos) empleando una metodología iterativa en la que cada nuevo modelo aprende de los resultados obtenidos por los modelos previamente entrenados. El proceso inicia con el modelo Holt-Winters, cuyos parámetros se ajustan con base en los hallazgos del análisis exploratorio y la descomposición STL de la serie temporal. Posteriormente, se incorporan secuencialmente los modelos ARIMA, SARIMA, Dynamic Optimized Theta, Ridge, Lasso, Random Forest, XGBoost, LightGBM y KNN, de modo que cada uno aprovecha el conocimiento acumulado por sus predecesores, con el objetivo de mejorar progresivamente el rendimiento predictivo. Las métricas de evaluación utilizadas son RMSE, MAE, MAPE, SMAPE y MSE. Finalmente, se selecciona el mejor modelo para realizar pronósticos del volumen de infracciones por estacionamiento prohibido para el año 2026, cuyos datos aún no han sido publicados por la Alcaldía de Barranquilla a través de su portal de Datos Abiertos.

## Carga de librerías

Se importan las librerías necesarias para el desarrollo de modelos predictivos sobre las series temporales de comparendos electrónicos. Se emplean `pandas` y `numpy` para la manipulación y transformación de datos, `plotly.graph_objects` y `plotly.subplots` para la visualización de resultados, y `statsmodels.tsa` para los modelos estadísticos clásicos (Holt-Winters, ARIMA, SARIMA y STL). Para los enfoques de machine learning, se utilizan `scikit-learn` con `RidgeCV`, `MultiTaskLassoCV`, `RandomForestRegressor`, `KNeighborsRegressor` y `GridSearchCV` para optimización de hiperparámetros, junto con `xgboost` (XGBoost) y `lightgbm` (LightGBM). Adicionalmente, se incorporan `sklearn.model_selection.TimeSeriesSplit` para validación con series temporales, `sklearn.preprocessing.StandardScaler` para estandarización de características, y `sklearn.metrics` para las métricas de evaluación (RMSE, MAE, MAPE, SMAPE y MSE). Se suprimen los mensajes de advertencia para mantener una salida limpia y enfocada en los resultados.

In [1]:
import warnings

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from scipy import stats as sp_stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

In [2]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [3]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [4]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [5]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [6]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [7]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Preparación de la Serie Temporal Mensual

Se extrae y configura la serie temporal mensual correspondiente a la infracción C02 (estacionamiento en sitios prohibidos) a partir del DataFrame de comparendos electrónicos. Para ello, se agrupan los registros por mes y código de infracción, sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por período. La serie resultante se indexa con fechas mensuales, se establece una frecuencia mensual ('MS') y se rellenan los valores faltantes con cero, asegurando una secuencia temporal continua y completa para los modelos predictivos. Se imprime información básica de la serie: número de meses, rango temporal y total de comparendos del período.

In [8]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):
    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_c02 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'C02')
print(f"Serie C02: {len(serie_c02)} meses")
print(f"Desde: {serie_c02.index.min().strftime('%Y-%m')} hasta: {serie_c02.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_c02.sum():,.0f}")

Serie C02: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 204,973


## División de la Serie Temporal en Conjuntos de Entrenamiento y Prueba

Se particiona la serie temporal mensual de la infracción C02 (estacionamiento en sitios prohibidos) en dos conjuntos: entrenamiento y prueba. El conjunto de entrenamiento comprende los meses desde enero de 2018 hasta diciembre de 2024, abarcando 84 meses, mientras que el conjunto de prueba corresponde al año completo de 2025, con 12 meses. Esta división permite entrenar los modelos predictivos con los datos históricos y evaluar su desempeño pronosticando el período más reciente (2025), cuyos valores reales ya están disponibles en la base de datos para calcular las métricas de error. Se imprime información detallada de ambos conjuntos, incluyendo su rango temporal y número de meses.

In [9]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_c02 = serie_c02[train_start:train_end].copy()
test_c02 = serie_c02[test_start:test_end].copy()

print(f"Train: {train_c02.index.min().strftime('%Y-%m')} a {train_c02.index.max().strftime('%Y-%m')} ({len(train_c02)} meses)")
print(f"Test: {test_c02.index.min().strftime('%Y-%m')} a {test_c02.index.max().strftime('%Y-%m')} ({len(test_c02)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


## Generación de Ventanas de Validación Cruzada Temporal

Se implementa una función de validación cruzada específica para series temporales, que respeta el orden cronológico de los datos y evita la fuga de información del futuro. La función `time_series_cv_manual` genera un conjunto de ventanas de entrenamiento y validación, donde el tamaño de cada ventana de validación es de 12 meses (un año completo). Se configuran 4 ventanas de validación, lo que significa que el modelo se evalúa con los últimos 4 años de datos disponibles. Para la infracción C02, se imprimen los rangos temporales de cada fold, mostrando el período de entrenamiento y validación correspondiente. Esta metodología permite evaluar el desempeño predictivo de los modelos en diferentes horizontes temporales de manera robusta y realista.

In [10]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):
    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_c02 = time_series_cv_manual(train_c02, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


## Holt-Winters

In [11]:
def evaluar_holt_winters(train_fold, val_fold, trend=None, seasonal=None,
                         damped_trend=False, seasonal_periods=None):
    """
    Ajusta un modelo de suavizamiento exponencial (Holt-Winters)
    con los componentes especificados.
    """
    modelo = ExponentialSmoothing(
        train_fold,
        trend=trend,
        seasonal=seasonal,
        damped_trend=damped_trend,
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [12]:
configuraciones_hw = [
    {'trend': None, 'seasonal': None, 'damped': False, 'nombre': 'SES (Simple)'},
    {'trend': 'add', 'seasonal': None, 'damped': False, 'nombre': 'Holt lineal'},
    {'trend': 'add', 'seasonal': None, 'damped': True,  'nombre': 'Holt amortiguado'},
    {'trend': None, 'seasonal': 'add', 'damped': False, 'nombre': 'Estacional aditiva'},
    {'trend': 'add', 'seasonal': 'add', 'damped': False, 'nombre': 'Holt‑Winters aditivo'},
]

resultados_cv_c02 = []

print("-" * 50)
print("Validación cruzada - Holt-Winters (C02)")
print("-" * 50)

for config in configuraciones_hw:
    nombre = config['nombre']
    print(f"\n--- {nombre} ---")
    
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
        pred, met, _ = evaluar_holt_winters(
            train_fold, val_fold,
            trend=config['trend'],
            seasonal=config['seasonal'],
            damped_trend=config['damped'],
            seasonal_periods=12 if config['seasonal'] else None
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_c02.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_c02 = pd.DataFrame(resultados_cv_c02)
mejor_idx = df_cv_c02['RMSE'].idxmin()
mejor_nombre = df_cv_c02.loc[mejor_idx, 'modelo']
mejor_config = next(c for c in configuraciones_hw if c['nombre'] == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo Holt‑Winters según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_c02.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Holt-Winters (C02)
--------------------------------------------------

--- SES (Simple) ---
  Fold 1: RMSE=929.54, MAE=889.25, MAPE=75.30%, SMAPE=51.91%, MSE=864045.25
  Fold 2: RMSE=517.23, MAE=438.00, MAPE=26.37%, SMAPE=31.60%, MSE=267531.83
  Fold 3: RMSE=1626.68, MAE=1477.59, MAPE=11223.08%, SMAPE=79.36%, MSE=2646099.64
  Fold 4: RMSE=421.95, MAE=365.47, MAPE=15.31%, SMAPE=13.84%, MSE=178041.15
  Promedio -> RMSE=873.85, MAE=792.58, MAPE=2835.02%, SMAPE=44.18%, MSE=988929.47

--- Holt lineal ---
  Fold 1: RMSE=764.22, MAE=678.61, MAPE=56.15%, SMAPE=41.54%, MSE=584024.66
  Fold 2: RMSE=802.32, MAE=720.07, MAPE=44.69%, SMAPE=60.50%, MSE=643711.00
  Fold 3: RMSE=1838.40, MAE=1664.98, MAPE=10983.81%, SMAPE=92.00%, MSE=3379732.88
  Fold 4: RMSE=390.84, MAE=337.57, MAPE=14.13%, SMAPE=12.88%, MSE=152755.28
  Promedio -> RMSE=948.94, MAE=850.31, MAPE=2774.70%, SMAPE=51.73%, MSE=1190055.96

--- Holt amortiguado ---
  Fo

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,SES (Simple),873.85,792.58,2835.02,44.18,988929.47
2,Holt amortiguado,904.50,819.62,2813.75,47.23,1057082.14
1,Holt lineal,948.94,850.31,2774.70,51.73,1190055.96
3,Estacional aditiva,963.91,849.21,2075.98,51.08,1116828.56
4,Holt‑Winters aditivo,1084.89,955.47,1968.91,66.81,1528321.23


In [13]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_c02 = ExponentialSmoothing(
    train_c02,
    trend=mejor_config['trend'],
    seasonal=mejor_config['seasonal'],
    damped_trend=mejor_config['damped'],
    seasonal_periods=12 if mejor_config['seasonal'] else None,
    initialization_method='estimated'
).fit()

pred_test_c02 = modelo_final_c02.forecast(len(test_c02))

real_test = test_c02.values
pred_test = pred_test_c02.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SES (Simple)
--------------------------------------------------
RMSE: 351.93
MAE: 280.29
MAPE: 13.67%
SMAPE: 12.75%
MSE: 123854.16


In [14]:
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_c02.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_c02.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_c02.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## ARIMA

In [15]:
def evaluar_arima(train_fold, val_fold, order=(1,0,0)):
    """
    Ajusta un modelo ARIMA con el orden (p,d,q) dado.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = ARIMA(train_fold, order=order)
    modelo_ajustado = modelo.fit()
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [16]:
ordenes_arima = [
    ((1,0,0), 'ARIMA(1,0,0)'),
    ((2,0,0), 'ARIMA(2,0,0)'),
    ((3,0,0), 'ARIMA(3,0,0)'),
    ((0,0,1), 'ARIMA(0,0,1)'),
    ((0,0,2), 'ARIMA(0,0,2)'),
    ((1,0,1), 'ARIMA(1,0,1)'),
    ((2,0,1), 'ARIMA(2,0,1)'),
    ((1,0,2), 'ARIMA(1,0,2)'),
    ((1,1,0), 'ARIMA(1,1,0)'),
    ((1,1,1), 'ARIMA(1,1,1)'),
    ((0,1,1), 'ARIMA(0,1,1)'),
]

resultados_cv_arima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos ARIMA (C02)")
print("-" * 50)

for orden, nombre in ordenes_arima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
        predicciones, met, _ = evaluar_arima(train_fold, val_fold, order=orden)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_arima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_arima = pd.DataFrame(resultados_cv_arima)
mejor_idx = df_cv_arima['RMSE'].idxmin()
mejor_orden, mejor_nombre = ordenes_arima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo ARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_arima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos ARIMA (C02)
--------------------------------------------------

--- ARIMA(1,0,0) ---
  Fold 1: RMSE=1002.42, MAE=965.82, MAPE=82.65%, SMAPE=55.14%, MSE=1004836.49
  Fold 2: RMSE=361.53, MAE=302.39, MAPE=21.20%, SMAPE=18.39%, MSE=130704.14
  Fold 3: RMSE=1245.21, MAE=1118.18, MAPE=12304.19%, SMAPE=58.41%, MSE=1550551.39
  Fold 4: RMSE=336.38, MAE=267.13, MAPE=10.07%, SMAPE=10.88%, MSE=113153.35
  Promedio -> RMSE=736.38, MAE=663.38, MAPE=3104.53%, SMAPE=35.71%, MSE=699811.34

--- ARIMA(2,0,0) ---
  Fold 1: RMSE=1010.61, MAE=973.30, MAPE=83.37%, SMAPE=55.43%, MSE=1021336.15
  Fold 2: RMSE=400.21, MAE=342.30, MAPE=24.08%, SMAPE=20.53%, MSE=160166.91
  Fold 3: RMSE=1219.42, MAE=1095.53, MAPE=12116.83%, SMAPE=57.38%, MSE=1486979.23
  Fold 4: RMSE=396.97, MAE=328.16, MAPE=12.43%, SMAPE=13.60%, MSE=157586.12
  Promedio -> RMSE=756.80, MAE=684.82, MAPE=3059.18%, SMAPE=36.74%, MSE=706517.10

--- ARIMA

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,"ARIMA(1,0,0)",736.38,663.38,3104.53,35.71,699811.34
9,"ARIMA(1,1,1)",751.92,674.69,2956.25,36.52,821224.37
5,"ARIMA(1,0,1)",752.95,680.71,3062.27,36.54,705631.69
2,"ARIMA(3,0,0)",754.86,682.15,3056.80,36.63,706546.66
6,"ARIMA(2,0,1)",756.68,684.53,3058.77,36.72,707372.77
1,"ARIMA(2,0,0)",756.80,684.82,3059.18,36.74,706517.10
7,"ARIMA(1,0,2)",770.07,694.59,2895.02,37.36,701048.15
4,"ARIMA(0,0,2)",802.58,718.84,3018.51,38.23,733300.11
3,"ARIMA(0,0,1)",845.91,777.45,3572.66,40.91,811122.37
10,"ARIMA(0,1,1)",871.71,790.18,2856.27,43.89,981880.90


In [17]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_arima = ARIMA(train_c02, order=mejor_orden)
modelo_final_arima_ajustado = modelo_final_arima.fit()

pred_test_arima = modelo_final_arima_ajustado.forecast(steps=len(test_c02))

real_test = test_c02.values
pred_test = pred_test_arima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - ARIMA(1,0,0)
--------------------------------------------------
RMSE: 380.94
MAE: 325.41
MAPE: 14.88%
SMAPE: 14.79%
MSE: 145116.53


In [18]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_arima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## SARIMA

In [19]:
def evaluar_sarima(train_fold, val_fold, order=(1,0,0), seasonal_order=(0,0,0,12)):
    """
    Ajusta un modelo SARIMAX con los órdenes especificados.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = SARIMAX(
        train_fold,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [20]:
configuraciones_sarima = [
    ((1,0,0), (0,0,0,12), 'SARIMA(1,0,0)(0,0,0)[12]'),
    ((1,0,0), (1,0,0,12), 'SARIMA(1,0,0)(1,0,0)[12]'),
    ((1,0,0), (0,0,1,12), 'SARIMA(1,0,0)(0,0,1)[12]'),
    ((1,0,0), (1,0,1,12), 'SARIMA(1,0,0)(1,0,1)[12]'),
    ((2,0,0), (0,0,0,12), 'SARIMA(2,0,0)(0,0,0)[12]'),
    ((2,0,0), (1,0,0,12), 'SARIMA(2,0,0)(1,0,0)[12]'),
    ((0,0,1), (0,0,0,12), 'SARIMA(0,0,1)(0,0,0)[12]'),
    ((0,0,1), (1,0,0,12), 'SARIMA(0,0,1)(1,0,0)[12]'),
    ((1,0,1), (0,0,0,12), 'SARIMA(1,0,1)(0,0,0)[12]'),
    ((1,0,1), (1,0,0,12), 'SARIMA(1,0,1)(1,0,0)[12]'),
    ((0,0,1), (0,1,1,12), 'SARIMA(0,0,1)(0,1,1)[12]'),
    ((1,0,0), (0,1,1,12), 'SARIMA(1,0,0)(0,1,1)[12]'),
    ((1,0,0), (1,0,0,12), 'SARIMA(1,0,0)(1,0,0)[12]'),
]

resultados_cv_sarima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos SARIMA (C02)")
print("-" * 50)

for order, seasonal_order, nombre in configuraciones_sarima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
        predicciones, met, _ = evaluar_sarima(
            train_fold, val_fold,
            order=order,
            seasonal_order=seasonal_order
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_sarima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_sarima = pd.DataFrame(resultados_cv_sarima)
mejor_idx = df_cv_sarima['RMSE'].idxmin()
mejor_nombre = df_cv_sarima.loc[mejor_idx, 'modelo']
mejor_order, mejor_seasonal = None, None
for order, seasonal_order, nombre in configuraciones_sarima:
    if nombre == mejor_nombre:
        mejor_order, mejor_seasonal = order, seasonal_order
        break

print("\n" + "-" * 50)
print("Mejor modelo SARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_sarima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos SARIMA (C02)
--------------------------------------------------

--- SARIMA(1,0,0)(0,0,0)[12] ---
  Fold 1: RMSE=662.19, MAE=501.73, MAPE=36.70%, SMAPE=31.40%, MSE=438495.91
  Fold 2: RMSE=903.21, MAE=821.39, MAPE=51.33%, SMAPE=72.66%, MSE=815784.41
  Fold 3: RMSE=2074.63, MAE=1874.16, MAPE=10529.09%, SMAPE=108.30%, MSE=4304074.02
  Fold 4: RMSE=505.28, MAE=436.94, MAPE=17.29%, SMAPE=19.53%, MSE=255312.47
  Promedio -> RMSE=1036.33, MAE=908.55, MAPE=2658.60%, SMAPE=57.97%, MSE=1453416.70

--- SARIMA(1,0,0)(1,0,0)[12] ---
  Fold 1: RMSE=1010.25, MAE=871.92, MAPE=76.57%, SMAPE=49.31%, MSE=1020609.60
  Fold 2: RMSE=941.16, MAE=849.45, MAPE=52.99%, SMAPE=76.50%, MSE=885773.63
  Fold 3: RMSE=2049.63, MAE=1852.63, MAPE=10603.24%, SMAPE=106.51%, MSE=4200966.94
  Fold 4: RMSE=378.44, MAE=317.51, MAPE=12.38%, SMAPE=13.53%, MSE=143220.26
  Promedio -> RMSE=1094.87, MAE=972.88, MAPE=2686.29%, SMAPE=61.4

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
11,"SARIMA(1,0,0)(0,1,1)[12]",765.33,668.16,3083.27,36.47,7.568888e+05
10,"SARIMA(0,0,1)(0,1,1)[12]",875.13,744.85,2885.87,40.39,8.780729e+05
4,"SARIMA(2,0,0)(0,0,0)[12]",1019.35,893.26,2659.44,56.84,1.419956e+06
8,"SARIMA(1,0,1)(0,0,0)[12]",1028.07,901.83,2647.93,57.50,1.437521e+06
0,"SARIMA(1,0,0)(0,0,0)[12]",1036.33,908.55,2658.60,57.97,1.453417e+06
12,"SARIMA(1,0,0)(1,0,0)[12]",1094.87,972.88,2686.29,61.46,1.562643e+06
1,"SARIMA(1,0,0)(1,0,0)[12]",1094.87,972.88,2686.29,61.46,1.562643e+06
9,"SARIMA(1,0,1)(1,0,0)[12]",1114.97,992.94,2623.42,64.57,1.610517e+06
5,"SARIMA(2,0,0)(1,0,0)[12]",1136.81,1007.90,2586.86,69.32,1.674892e+06
7,"SARIMA(0,0,1)(1,0,0)[12]",1304.67,1122.75,2229.63,84.85,1.856795e+06


In [21]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_sarima = SARIMAX(
    train_c02,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
modelo_final_sarima_ajustado = modelo_final_sarima.fit(disp=False)

pred_test_sarima = modelo_final_sarima_ajustado.forecast(steps=len(test_c02))

real_test = test_c02.values
pred_test = pred_test_sarima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SARIMA(1,0,0)(0,1,1)[12]
--------------------------------------------------
RMSE: 402.08
MAE: 353.72
MAPE: 15.66%
SMAPE: 16.36%
MSE: 161669.03


In [22]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=train_c02.index, y=train_c02.values, mode='lines+markers', name='Train (2018-2024)',
                           line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
                           hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'))
fig1.add_trace(go.Scatter(x=test_c02.index, y=test_c02.values, mode='lines+markers', name='Test Real (2025)',
                           line=dict(color='green', width=2), marker=dict(size=4),
                           hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'))
fig1.add_trace(go.Scatter(x=test_c02.index, y=pred_test_sarima.values, mode='lines+markers',
                           name=f'Predicción {mejor_nombre}',
                           line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
                           hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'))
fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>', showarrow=False,
                    font=dict(size=12, color='darkgray'), xanchor='left', yanchor='bottom')
fig1.update_layout(title=dict(text=f'C02 - {mejor_nombre}: Predicción vs Real<br>'
                                   f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
                              x=0.5, font=dict(size=16, weight='bold')),
                   xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
                   yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
                   template='plotly_white', hovermode='x unified',
                   legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)', bordercolor='lightgray', borderwidth=1),
                   width=1055, height=500)
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=meses, y=test_c02.values, mode='lines+markers', name='Real',
                           line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
                           hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'))
fig2.add_trace(go.Scatter(x=meses, y=pred_test_sarima.values, mode='lines+markers',
                           name=f'Predicción {mejor_nombre}',
                           line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
                           hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'))
fig2.update_layout(title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
                   xaxis_title=dict(text='Mes', font=dict(weight='bold')),
                   yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
                   template='plotly_white', hovermode='x unified',
                   legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)', bordercolor='lightgray', borderwidth=1),
                   width=1055, height=500)
fig2.show()

fig3 = go.Figure()
errores_test = test_c02.values - pred_test_sarima.values
fig3.add_trace(go.Bar(x=meses, y=errores_test, name='Error (Real - Predicción)',
                      marker_color='mediumpurple', marker_line_color='darkorchid',
                      marker_line_width=1, opacity=1,
                      hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'))
fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)
fig3.update_layout(title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
                   xaxis_title=dict(text='Mes', font=dict(weight='bold')),
                   yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
                   template='plotly_white', hovermode='x unified',
                   showlegend=False, width=1055, height=500)
fig3.show()

## Dynamic Optimized Theta

In [23]:
def evaluar_theta(train_fold, val_fold, theta=2.0, period=12):
    """
    Ajusta un modelo Theta con el parámetro theta dado.
    Procedimiento:
      1. Descomponer la serie con STL (period=12) para obtener estacionalidad.
      2. Serie desestacionalizada = serie - estacionalidad.
      3. Ajustar regresión lineal a la serie desestacionalizada (tendencia global).
      4. Ajustar SES a la serie desestacionalizada.
      5. Combinar pronósticos: Yhat = SES_forecast + theta*(trend_forecast - SES_forecast).
      6. Sumar estacionalidad (último año).
    Retorna predicciones, métricas y un diccionario con información del modelo.
    """
    
    if len(train_fold) < 2 * period:
        ses = ExponentialSmoothing(train_fold, trend=None, seasonal=None,
                                   initialization_method='estimated').fit()
        pred = ses.forecast(len(val_fold))
        real = val_fold.values
        pred = pred.values
        mse = mean_squared_error(real, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(real, pred)
        mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
        smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
        return pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}, {'theta': theta, 'method': 'SES'}
    
    stl = STL(train_fold, period=period, robust=True).fit()
    seasonal = stl.seasonal
    deseasonalized = train_fold - seasonal

    x = np.arange(len(deseasonalized))
    slope, intercept, _, _, _ = sp_stats.linregress(x, deseasonalized.values)
    trend_line = slope * x + intercept

    ses_model = ExponentialSmoothing(deseasonalized, trend=None, seasonal=None,
                                     initialization_method='estimated').fit()
    
    h = len(val_fold)
    ses_forecast = ses_model.forecast(h)
    last_idx = len(deseasonalized) - 1
    trend_extrapolated = slope * np.arange(last_idx + 1, last_idx + 1 + h) + intercept

    theta_forecast = ses_forecast.values + theta * (trend_extrapolated - ses_forecast.values)

    last_year_seasonal = seasonal.iloc[-period:].values
    repeats = int(np.ceil(h / period))
    seasonal_pattern = np.tile(last_year_seasonal, repeats)[:h]
    final_forecast = theta_forecast + seasonal_pattern

    real = val_fold.values
    pred = final_forecast

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}
    modelo_info = {'theta': theta, 'slope': slope, 'intercept': intercept}
    
    return final_forecast, metricas, modelo_info

In [24]:
thetas = np.arange(0.0, 4.0 + 0.25, 0.25)

resultados_cv_theta = []

print("-" * 50)
print("Validación cruzada - Optimización de Theta (C02)")
print("-" * 50)

for theta in thetas:
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
        _, met, _ = evaluar_theta(train_fold, val_fold, theta=theta, period=12)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_theta.append({
        'theta': theta,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })

df_cv_theta = pd.DataFrame(resultados_cv_theta)
mejor_idx = df_cv_theta['RMSE'].idxmin()
mejor_theta = df_cv_theta.loc[mejor_idx, 'theta']
mejor_rmse_cv = df_cv_theta.loc[mejor_idx, 'RMSE']

print("\n--- Resultados de optimización de theta ---")
print(f"Mejor theta: {mejor_theta:.2f} (RMSE CV promedio = {mejor_rmse_cv:.2f})")
print("\nTop 5 configuraciones:")
display(df_cv_theta.sort_values('RMSE').head())

print("\n" + "-" * 50)
print(f"Validación cruzada - Theta dinámico óptimo (θ={mejor_theta:.2f})")
print("-" * 50)

metricas_opt_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    _, met, _ = evaluar_theta(train_fold, val_fold, theta=mejor_theta, period=12)
    metricas_opt_folds['RMSE'].append(met['RMSE'])
    metricas_opt_folds['MAE'].append(met['MAE'])
    metricas_opt_folds['MAPE'].append(met['MAPE'])
    metricas_opt_folds['SMAPE'].append(met['SMAPE'])
    metricas_opt_folds['MSE'].append(met['MSE'])
    print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
          f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

prom_opt = {k: np.mean(v) for k, v in metricas_opt_folds.items()}
print(f"  Promedio -> RMSE={prom_opt['RMSE']:.2f}, MAE={prom_opt['MAE']:.2f}, "
      f"MAPE={prom_opt['MAPE']:.2f}%, SMAPE={prom_opt['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Optimización de Theta (C02)
--------------------------------------------------

--- Resultados de optimización de theta ---
Mejor theta: 0.25 (RMSE CV promedio = 1255.68)

Top 5 configuraciones:


,theta,RMSE,MAE,MAPE,SMAPE,MSE
1,0.25,1255.678069,1104.006800,2617.774779,75.319422,1.668881e+06
0,0.00,1272.420434,1121.241865,2935.650754,72.428431,1.670627e+06
2,0.50,1289.652342,1113.137267,2301.107560,79.576558,1.793189e+06
3,0.75,1384.920826,1184.649837,1986.950212,87.091153,2.043549e+06
4,1.00,1520.848561,1316.482823,1675.256074,97.907987,2.419963e+06



--------------------------------------------------
Validación cruzada - Theta dinámico óptimo (θ=0.25)
--------------------------------------------------
  Fold 1: RMSE=1551.14, MAE=1392.76, MAPE=104.13%, SMAPE=126.09%, MSE=2406030.88
  Fold 2: RMSE=1040.66, MAE=857.19, MAPE=53.54%, SMAPE=69.87%, MSE=1082969.85
  Fold 3: RMSE=1555.94, MAE=1405.24, MAPE=10283.18%, SMAPE=76.63%, MSE=2420934.25
  Fold 4: RMSE=874.98, MAE=760.84, MAPE=30.25%, SMAPE=28.69%, MSE=765590.29
  Promedio -> RMSE=1255.68, MAE=1104.01, MAPE=2617.77%, SMAPE=75.32%, MSE=16769092.81


In [25]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - Theta optimizado (θ={mejor_theta:.2f})")
print("-" * 50)

pred_test_theta, met_train, modelo_info = evaluar_theta(train_c02, test_c02, theta=mejor_theta, period=12)

real_test = test_c02.values
pred_test = pred_test_theta

mse_test = met_train['MSE']
rmse_test = met_train['RMSE']
mae_test = met_train['MAE']
mape_test = met_train['MAPE']
smape_test = met_train['SMAPE']

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Theta optimizado (θ=0.25)
--------------------------------------------------
RMSE: 383.33
MAE: 307.51
MAPE: 14.69%
SMAPE: 14.20%
MSE: 146945.14


In [26]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - Theta Dinámico Optimizado (θ={mejor_theta:.2f}): Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_theta

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Ridge

In [27]:
def crear_dataset_supervisado_ridge(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas:
    - Tendencia normalizada
    - Componentes cíclicos del mes (seno/coseno)
    - Dummy COVID (1 si el mes a predecir cae entre marzo y diciembre de 2020)
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_ridge(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta RidgeCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    
    X_train, y_train = crear_dataset_supervisado_ridge(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = RidgeCV(alphas=alphas, cv=tscv)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0  
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [28]:
resultados_cv_ridge = []

print("-" * 50)
print("Validación cruzada - Ridge Regression (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_ridge(train_fold, val_fold)
    
    resultados_cv_ridge.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_ridge = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ridge]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ridge]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ridge]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ridge]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ridge])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_ridge['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_ridge['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_ridge['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_ridge['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_ridge['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Ridge Regression (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 1.00
RMSE: 2471.36
MAE: 2051.38
MAPE: 163.52%
SMAPE: 150.20%
MSE: 6107625.26

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 1000.00
RMSE: 427.18
MAE: 362.70
MAPE: 25.53%
SMAPE: 22.45%
MSE: 182487.02

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 100.00
RMSE: 1598.68
MAE: 1470.96
MAPE: 12605.06%
SMAPE: 78.65%
MSE: 2555778.08

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 1000.00
RMSE: 951.03
MAE: 907.93
MAPE: 35.01%
SMAPE: 43.01%
MSE: 904466.24

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 1362.07
MAE: 1198.25
MAPE: 3207.28%
SMAPE: 73.58%
MSE: 2437589

In [29]:
print("-" * 50)
print("Evaluación final en Test (2025) - Ridge Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_ridge(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
modelo_final_ridge = RidgeCV(alphas=alphas, cv=tscv)
modelo_final_ridge.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_ridge.alpha_:.2f}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0 
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_ridge = modelo_final_ridge.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_ridge

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Ridge Regression
--------------------------------------------------
Alpha seleccionado: 1000.00
RMSE: 522.39
MAE: 477.87
MAPE: 20.37%
SMAPE: 22.16%
MSE: 272890.85


In [30]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - Ridge Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_ridge

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Lasso

In [31]:
def crear_dataset_supervisado_lasso(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lasso(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta MultiTaskLassoCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = np.logspace(-4, 4, 50)
    
    X_train, y_train = crear_dataset_supervisado_lasso(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [32]:
resultados_cv_lasso = []

print("-" * 50)
print("Validación cruzada - Lasso Regression (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lasso(train_fold, val_fold)
    
    resultados_cv_lasso.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lasso = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lasso]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lasso]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lasso]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lasso]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lasso])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lasso['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lasso['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lasso['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lasso['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lasso['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Lasso Regression (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 1048.11
RMSE: 1576.72
MAE: 1417.92
MAPE: 115.76%
SMAPE: 122.71%
MSE: 2486044.99

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 10000.00
RMSE: 433.05
MAE: 363.63
MAPE: 25.87%
SMAPE: 22.40%
MSE: 187529.33

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 10000.00
RMSE: 1534.43
MAE: 1370.09
MAPE: 14575.63%
SMAPE: 70.61%
MSE: 2354480.96

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 10000.00
RMSE: 927.92
MAE: 892.80
MAPE: 34.50%
SMAPE: 42.07%
MSE: 861030.43

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 1118.03
MAE: 1011.11
MAPE: 3687.94%
SMAPE: 64.45%
MSE: 

In [33]:
print("-" * 50)
print("Evaluación final en Test (2025) - Lasso Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lasso(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = np.logspace(-4, 4, 50)
modelo_final_lasso = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
modelo_final_lasso.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_lasso.alpha_:.2f}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lasso = modelo_final_lasso.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_lasso

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Lasso Regression
--------------------------------------------------
Alpha seleccionado: 1526.42
RMSE: 557.95
MAE: 511.89
MAPE: 21.64%
SMAPE: 23.91%
MSE: 311309.35


In [34]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - Lasso Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_lasso

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Random Forest

In [35]:
def crear_dataset_supervisado_rf(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_random_forest(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta Random Forest multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [5, 10, 15, None],
            'estimator__min_samples_split': [2, 5, 10]
        }
    
    X_train, y_train = crear_dataset_supervisado_rf(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    rf = RandomForestRegressor(random_state=42)
    multi_rf = MultiOutputRegressor(rf)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_rf, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [36]:
resultados_cv_rf = []

print("-" * 50)
print("Validación cruzada - Random Forest (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_random_forest(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    min_samples_split = best_params['estimator__min_samples_split']
    
    resultados_cv_rf.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_rf = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_rf]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_rf]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_rf]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_rf]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_rf])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_rf['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_rf['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_rf['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_rf['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_rf['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Random Forest (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=200, max_depth=5, min_samples_split=2
RMSE: 757.75
MAE: 673.68
MAPE: 51.13%
SMAPE: 48.05%
MSE: 574180.26

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=300, max_depth=5, min_samples_split=10
RMSE: 443.84
MAE: 364.11
MAPE: 23.40%
SMAPE: 23.98%
MSE: 196991.84

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=10, min_samples_split=10
RMSE: 1541.43
MAE: 1417.68
MAPE: 10361.06%
SMAPE: 76.61%
MSE: 2375999.16

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=300, max_depth=10, min_samples_split=10
RMSE: 1224.82
MAE: 1123.40
MAPE: 43.30%
SMAPE: 57.30%
MSE: 1500195.94

----------------------------------

In [37]:
print("-" * 50)
print("Evaluación final en Test (2025) - Random Forest")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_rf(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [5, 10, 15, None],
    'estimator__min_samples_split': [2, 5, 10]
}

rf_final = RandomForestRegressor(random_state=42)
multi_rf_final = MultiOutputRegressor(rf_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_rf_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_rf = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0 
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_rf = modelo_final_rf.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_rf

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Random Forest
--------------------------------------------------
Mejores parámetros: {'estimator__max_depth': 10, 'estimator__min_samples_split': 10, 'estimator__n_estimators': 100}
RMSE: 535.89
MAE: 421.74
MAPE: 21.37%
SMAPE: 17.82%
MSE: 287179.21


In [38]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - Random Forest: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_rf

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## XGBoost

In [39]:
def crear_dataset_supervisado_xgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_xgboost(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta XGBoost multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_xgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    xgb = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
    multi_xgb = MultiOutputRegressor(xgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [40]:
resultados_cv_xgb = []

print("-" * 50)
print("Validación cruzada - XGBoost (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_xgboost(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_xgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_xgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_xgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_xgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_xgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_xgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_xgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_xgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_xgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_xgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_xgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_xgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - XGBoost (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=300, max_depth=7, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8
RMSE: 886.81
MAE: 733.70
MAPE: 54.78%
SMAPE: 70.13%
MSE: 786427.19

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=1.0, colsample_bytree=0.8
RMSE: 567.01
MAE: 472.95
MAPE: 32.12%
SMAPE: 28.51%
MSE: 321499.13

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 1514.97
MAE: 1376.41
MAPE: 11589.48%
SMAPE: 73.02%
MSE: 2295136.29

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample

In [41]:
print("-" * 50)
print("Evaluación final en Test (2025) - XGBoost")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_xgb(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

xgb_final = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
multi_xgb_final = MultiOutputRegressor(xgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_xgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_xgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0 
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_xgb = modelo_final_xgb.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_xgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - XGBoost
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 5, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 382.19
MAE: 279.24
MAPE: 13.45%
SMAPE: 12.74%
MSE: 146072.47


In [42]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - XGBoost: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_xgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## LightGBM

In [43]:
def crear_dataset_supervisado_lgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lightgbm(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta LightGBM multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7, -1],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_lgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb = LGBMRegressor(random_state=42, verbosity=-1)
    multi_lgb = MultiOutputRegressor(lgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_lgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [44]:
resultados_cv_lgb = []

print("-" * 50)
print("Validación cruzada - LightGBM (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lightgbm(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_lgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - LightGBM (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 746.09
MAE: 570.88
MAPE: 52.85%
SMAPE: 34.88%
MSE: 556645.89

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 433.05
MAE: 363.63
MAPE: 25.87%
SMAPE: 22.40%
MSE: 187529.33

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 1534.43
MAE: 1370.09
MAPE: 14575.63%
SMAPE: 70.61%
MSE: 2354480.96

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsamp

In [45]:
print("-" * 50)
print("Evaluación final en Test (2025) - LightGBM")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lgb(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, -1],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

lgb_final = LGBMRegressor(random_state=42, verbosity=-1)
multi_lgb_final = MultiOutputRegressor(lgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_lgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_lgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0  
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lgb = modelo_final_lgb.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_lgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - LightGBM
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 382.45
MAE: 321.24
MAPE: 15.21%
SMAPE: 14.62%
MSE: 146268.63


In [46]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - LightGBM: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_lgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## KNN

In [47]:
def crear_dataset_supervisado_knn(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_knn(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta KNN multi‑output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_neighbors': [2, 3, 5, 7, 10],
            'estimator__weights': ['uniform', 'distance'],
            'estimator__p': [1, 2]
        }
    
    X_train, y_train = crear_dataset_supervisado_knn(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    knn = KNeighborsRegressor()
    multi_knn = MultiOutputRegressor(knn)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_knn, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [48]:
resultados_cv_knn = []

print("-" * 50)
print("Validación cruzada - KNN Regression (C02)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c02, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_knn(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_neighbors = best_params['estimator__n_neighbors']
    weights = best_params['estimator__weights']
    p = best_params['estimator__p']
    
    resultados_cv_knn.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_neighbors={n_neighbors}, weights={weights}, p={p}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_knn = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_knn]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_knn]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_knn]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_knn]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_knn])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_knn['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_knn['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_knn['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_knn['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_knn['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - KNN Regression (C02)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_neighbors=2, weights=distance, p=2
RMSE: 1147.24
MAE: 1118.28
MAPE: 90.57%
SMAPE: 112.58%
MSE: 1316163.04

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_neighbors=7, weights=uniform, p=1
RMSE: 492.05
MAE: 439.43
MAPE: 30.39%
SMAPE: 27.27%
MSE: 242111.32

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_neighbors=10, weights=uniform, p=1
RMSE: 1541.47
MAE: 1434.22
MAPE: 11655.40%
SMAPE: 76.92%
MSE: 2376120.46

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_neighbors=10, weights=uniform, p=1
RMSE: 921.89
MAE: 839.96
MAPE: 32.18%
SMAPE: 39.79%
MSE: 849888.22

--------------------------------------------------
Métricas promedio en validación cruzada
---

In [49]:
print("-" * 50)
print("Evaluación final en Test (2025) - KNN Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_knn(
    train_c02.values, train_c02.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_neighbors': [2, 3, 5, 7, 10, 12],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__p': [1, 2]
}

knn_final = KNeighborsRegressor()
multi_knn_final = MultiOutputRegressor(knn_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_knn_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_knn = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c02.values[-12:]
fecha_fin_pred_test = train_c02.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0  # 2025 no es COVID
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_knn = modelo_final_knn.predict(X_pred_test_scaled).flatten()

real_test = test_c02.values
pred_test = pred_test_knn

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c02.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - KNN Regression
--------------------------------------------------
Mejores parámetros: {'estimator__n_neighbors': 10, 'estimator__p': 1, 'estimator__weights': 'distance'}
RMSE: 408.24
MAE: 322.30
MAPE: 13.72%
SMAPE: 14.91%
MSE: 166656.13


In [50]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c02.index, y=train_c02.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=test_c02.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c02.index, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C02 - KNN Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c02.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c02.values - pred_test_knn

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Tabla 1 – Métricas Promedio en Validación Cruzada (CV)

| Modelo                     | RMSE      | MAE      | MAPE     | SMAPE   | MSE          |
|----------------------------|-----------|----------|----------|---------|--------------|
| Holt‑Winters (SES)         | 873.85    | 792.58   | 2835.02% | 44.18%  | 988929.47    |
| ARIMA(1,0,0)               | 736.38    | 663.38   | 3104.53% | 35.71%  | 699811.34    |
| SARIMA(1,0,0)(0,1,1)[12]   | 765.33    | 668.16   | 3083.27% | 36.47%  | 756888.85    |
| Theta (θ=0.25)             | 1255.68   | 1104.01  | 2617.77% | 75.32%  | 16769092.81  |
| Ridge                      | 1362.07   | 1198.25  | 3207.28% | 73.58%  | 2437589.15   |
| Lasso                      | 1118.03   | 1011.11  | 3687.94% | 64.45%  | 1472271.43   |
| Random Forest              | 991.96    | 894.72   | 2619.72% | 51.49%  | 1161841.80   |
| XGBoost                    | 1017.59   | 886.66   | 3090.16% | 54.54%  | 1162859.04   |
| LightGBM                   | 923.30    | 798.47   | 3672.10% | 42.64%  | 1014587.43   |
| KNN                        | 1025.66   | 957.97   | 2952.14% | 64.14%  | 1196070.76   |


## Tabla 2 – Métricas en Test (2025)

| Modelo                     | RMSE    | MAE     | MAPE    | SMAPE   | MSE        |
|----------------------------|---------|---------|---------|---------|------------|
| Holt‑Winters (SES)         | 351.93  | 280.29  | 13.67%  | 12.75%  | 123854.16  |
| ARIMA(1,0,0)               | 380.94  | 325.41  | 14.88%  | 14.79%  | 145116.53  |
| SARIMA(1,0,0)(0,1,1)[12]   | 402.08  | 353.72  | 15.66%  | 16.36%  | 161669.03  |
| Theta (θ=0.25)             | 383.33  | 307.51  | 14.69%  | 14.20%  | 146945.14  |
| Ridge                      | 522.39  | 477.87  | 20.37%  | 22.16%  | 272890.85  |
| Lasso                      | 557.95  | 511.89  | 21.64%  | 23.91%  | 311309.35  |
| Random Forest              | 535.89  | 421.74  | 21.37%  | 17.82%  | 287179.21  |
| XGBoost                    | 384.75  | 283.23  | 13.64%  | 12.89%  | 148034.17  |
| LightGBM                   | 382.45  | 321.24  | 15.21%  | 14.62%  | 146268.63  |
| KNN                        | 408.24  | 322.30  | 13.72%  | 14.91%  | 166656.13  |

## Tabla 3 – Errores por Mes en Test (2025)

| Modelo                     | Ene  | Feb  | Mar  | Abr  | May  | Jun  | Jul  | Ago   | Sep  | Oct  | Nov  | Dic   |
|----------------------------|------|------|------|------|------|------|------|-------|------|------|------|-------|
| Holt‑Winters (SES)         | 139  | 141  | 204  | 146  | 167  | -153 | 150  | -849  | 441  | 494  | 104  | -374  |
| ARIMA(1,0,0)               | 171  | 201  | 285  | 245  | 279  | -30  | 282  | -711  | 585  | 642  | 255  | -220  |
| SARIMA(1,0,0)(0,1,1)[12]   | 564  | 227  | 232  | 554  | 365  | 113  | 292  | -534  | 740  | 299  | 205  | -119  |
| Theta (θ=0.25)             | 239  | 517  | 261  | 256  | 55   | -135 | -154 | -818  | 688  | 137  | 158  | -271  |
| Ridge                      | 485  | 472  | 537  | 464  | 487  | 194  | 502  | -491  | 796  | 839  | 458  | -11   |
| Lasso                      | 531  | 509  | 563  | 493  | 519  | 226  | 544  | -440  | 857  | 905  | 519  | 36    |
| Random Forest              | 309  | -221 | -219 | -496 | -454 | -774 | -461 | -1278 | 25   | 61   | -206 | -556  |
| XGBoost                    | 236  | 38   | -81  | -94  | -97  | -272 | -44  | -885  | 482  | 664  | 384  | -122  |
| LightGBM                   | 364  | 126  | 327  | 251  | 298  | -47  | 254  | -854  | 496  | 469  | 88   | -281  |
| KNN                        | -49  | -165 | 40   | 79   | 304  | 183  | 619  | -284  | 804  | 725  | 384  | 231   |

## Interpretación de los Resultados de los Modelos

**a) Desempeño en validación cruzada (CV, 2018‑2024)**  
- Las métricas de CV presentan MAPE extremadamente elevadas (miles de %) y un RMSE que varía entre 736 y 1362. Estas cifras están distorsionadas por el Fold 3 (validación en 2023), donde la cantidad real de comparendos fue muy baja en algunos meses, haciendo que pequeños errores absolutos se tradujeran en porcentajes descomunales.  
- Descontando ese efecto, los modelos más cuidadosos ARIMA(1,0,0), SARIMA(1,0,0)(0,1,1)[12] y Holt‑Winters SES mostraron los menores RMSE promedio (736–873). Los métodos de machine learning (XGBoost, LightGBM, Random Forest) y los regularizados quedaron por encima de 900 de RMSE, señalando cierta inestabilidad en la validación.  
- La fuerte autocorrelación detectada en los residuos de la descomposición STL (Ljung‑Box significativo) indica que cualquier modelo debía incorporar una estructura autorregresiva. Por ello ARIMA(1,0,0) que es un AR(1) resultó competitivo en CV.

**b) Desempeño en el conjunto de prueba (2025)**  
- En 2025 todos los modelos tuvieron un desempeño razonable, con RMSE entre 352 y 558, MAPE de 13.6% a 21.6%. Esto es consistente con una serie estacionaria, sin tendencia y con estacionalidad débil, donde los valores futuros oscilan alrededor de una media constante.  
- Los mejores fueron Holt‑Winters SES (RMSE = 352, MAE = 280) y XGBoost (RMSE = 385, MAE = 283), seguidos muy de cerca por LightGBM (382), ARIMA(1,0,0) (381) y Theta optimizado (383). Las diferencias entre ellos son pequeñas.  
- El error más notable de todos los modelos ocurrió en agosto de 2025, donde se subestimó la cifra real (errores de -534 a -885). Todos los modelos “esperaban” un valor más bajo para ese mes, pero la realidad presentó un repunte atípico que ningún modelo pudo anticipar.  
- La mayoría de los errores mensuales son positivos (predicen por debajo del valor real) en los meses de enero a mayo y septiembre a noviembre, y negativos (sobrepredicen) en junio y diciembre, reflejando que la serie no sigue un patrón estacional marcado y que los modelos simplemente se anclan en el nivel medio.

---

## ¿Por qué se llegó a estos resultados?

**a) Naturaleza de la serie**  
- El análisis descriptivo y la descomposición STL revelaron que la serie es estacionaria (ADF p‑valor = 0.0066), no posee tendencia significativa (p = 0.28) y la estacionalidad explica solo el 5.2% de la varianza.  
- En una serie sin tendencia y con oscilaciones alrededor de una media, los modelos que proyectan un nivel constante o que capturan la dependencia temporal de corto plazo (autocorrelación) tienden a funcionar bien, mientras que los modelos que intentan aprender patrones complejos o tendencias inexistentes pueden sobreajustar.

**b) Desempeño en validación cruzada**  
- Los modelos ARIMA(1,0,0) y Holt‑Winters SES se beneficiaron de su simplicidad: un AR(1) captura la dependencia serial (confirmada por el ACF/PACF con un fuerte lag 1), y SES simplemente suaviza el nivel.  
- El SARIMA seleccionado incluye una diferencia estacional (0,1,1)[12], lo cual puede ayudar a manejar remanentes de estacionalidad, pero no ofreció una ventaja decisiva sobre los modelos más simples.  
- Los métodos de árboles y boosting sufrieron en el Fold 3 (2023) porque ese año tuvo valores inusualmente bajos que no estaban representados en el historial de entrenamiento, generando grandes errores porcentuales. La regularización de Ridge/Lasso también mostró inestabilidad.

**c) Desempeño en test (2025)**  
- Al ser el 2025 un año sin quiebres estructurales respecto al patrón histórico (solo el pico aislado de agosto), todos los modelos lograron un error absoluto razonable.  
- Holt‑Winters SES resultó ligeramente superior porque es extremadamente robusto frente a la no normalidad y a los outliers (la serie tiene residuos no normales), y porque al no intentar modelar una tendencia evita el sesgo que otros modelos podrían introducir.  
- XGBoost y LightGBM aprendieron las interacciones no lineales de la serie, pero al no haber una dinámica compleja que explotar, apenas igualan a los métodos clásicos. La pequeña ventaja de SES en RMSE se debe a que sus errores son más homogéneos mes a mes, mientras que los basados en árboles aciertan unos meses y fallan en otros (por ejemplo, XGBoost tuvo un error de -885 en agosto).

**d) Autocorrelación y residuos**  
- La estructura AR(1) presente en los residuos STL fue aprovechada directamente por ARIMA(1,0,0) e indirectamente por los árboles al incluir rezagos como variables. Pero como la autocorrelación no es extrema, un simple suavizado exponencial (SES) ya captura buena parte de la rutina temporal.

---

## Modelo Ideal para Predecir 2026

Se recomienda utilizar Holt‑Winters con suavizamiento exponencial simple (SES) para el pronóstico de 2026 del código C02.

**Justificación:**  
- Fue el modelo con mejor desempeño en el test de 2025 (RMSE=352, MAE=280, MAPE=13.7%), superando ligeramente a XGBoost y ARIMA.  
- Al ser estacionaria y sin tendencia, un pronóstico que se ancla en el nivel promedio reciente y se adapta lentamente es teóricamente el más adecuado. SES no impone una tendencia adulterada ni sobreajusta patrones ficticios.  
- Es un modelo fácil de mantener, poco sensible a valores atípicos y no requiere reoptimización constante de hiperparámetros. Dado que los residuos no son normales y existen eventos aislados (como el pico de agosto 2025), la simplicidad de SES lo hace más fiable a largo plazo.  

## Pronóstico de la Infracción C02 para el Año 2026

Una vez seleccionado el mejor modelo predictivo (Suavización Exponencial Simple, SES, sin tendencia ni componente estacional), se procede a realizar el pronóstico del volumen de infracciones por estacionamiento en sitios prohibidos para los 12 meses del año 2026. El modelo se entrena con la serie completa de 96 meses (enero 2018 a diciembre 2025) y genera predicciones puntuales constantes para cada mes de 2026, ya que el modelo SES sin tendencia asume que el futuro se comportará como el nivel promedio estimado del último período. Los resultados se visualizan mediante dos gráficos interactivos: el primero muestra la serie histórica completa junto con la proyección hacia 2026, con una línea vertical discontinua que marca el inicio del período pronosticado; el segundo presenta exclusivamente el pronóstico mensual para 2026, facilitando la interpretación de la estabilidad esperada mes a mes. Esta visualización permite anticipar el comportamiento futuro de la infracción y apoyar la toma de decisiones en materia de control de estacionamiento prohibido.

In [51]:
modelo_final_c02_2026 = ExponentialSmoothing(
    serie_c02,
    trend=None,           
    seasonal=None,
    damped_trend=False,
    initialization_method='estimated'
).fit()

forecast_2026_c02 = modelo_final_c02_2026.forecast(12)

idx_forecast = pd.date_range(start='2026-01-01', periods=12, freq='MS')
forecast_2026_c02.index = idx_forecast

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=serie_c02.index,
    y=serie_c02.values,
    mode='lines+markers',
    name='Histórico (2018-2025)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=forecast_2026_c02.index,
    y=forecast_2026_c02.values,
    mode='lines+markers',
    name='Predicción SES 2026',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2026-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2026-01-01',
    y=1,
    yref='paper',
    text='<b>Inicio 2026</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text='C02 - Suavización Exponencial Simple (SES): Predicción 2026<br>'
             '<sup>Modelo entrenado con datos hasta diciembre 2025</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()


meses_2026 = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
              'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses_2026,
    y=forecast_2026_c02.values,
    mode='lines+markers',
    name='Predicción SES 2026',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Pronóstico C02 para el año 2026',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig2.show()